### 1.1. Sets

A **set** in AMPL is a finite collection of distinct members(symbolic or numbers) used to index model components — parameters, variables, constraints, and summations. 

Sets are the most fundamental building block of an AMPL model: almost every parameter, variable, and constraint in a realistic model is indexed over at least one set.

**Sets serve three roles in every AMPL model:**
- **Indexing**: `param cost {PROD} >= 0;` declares one cost value per member of `PROD`.
- **Iteration**: `sum {p in PROD} revenue[p] * Sell[p]` sums over all members of `PROD`.
- **Constraint generation**: `subject to Limit {p in PROD}: Sell[p] <= market[p];` generates one constraint instance per member.

#### General Syntax

In [ ]:
set SET_NAME [indexing] [attributes] ;
   - set                       # AMPL keyword for declaring a set
   - SET_NAME                  # Name of the set
   - [{indexing}]              # optional: indexed family of sets (one set per index)
   - [attributes]:             # Optional: restrictions, ordering, definition, or default value
     - [dimen n]               # Tuple size: `dimen 1` = single values, `dimen 2` = pairs, `dimen 3` = triples 
     - [within SEXPR]          # Restricts members to a set expression
     - [ordered | circular]    # Keeps an order for a one-dimensional set (circular = wrap-around for `next()` and `prev()`)
     - [ordered by EXPR]       # Inherits order from another ordered set expression
     - [= SEXPR]               # Defines a computed set; data cannot override it
     - [default SEXPR]         # Gives a fallback value if no data is provided
     - [={} | default {}]      # Defines an empty set | Uses an empty set as fallback

#### What is `SEXPR`?
`SEXPR` means **set expression** — anything that produces a set.

Examples of set expressions:
| AMPL syntax / expression            | Meaning                                             |
| ----------------------------------- | --------------------------------------------------- |
| `{"bands", "coils", "plate"}`       | Explicit symbolic set                               |
| `1..12`                             | Numeric range                                       |
| `PROD`                              | Another set                                         |
| `A union B`                         | Union of two sets                                   |
| `A inter B`                         | Intersection of two sets                            |
| `A diff B`                          | Difference: elements in `A` but not in `B`          |
| `ORIG cross DEST`                   | Cartesian product, set of pairs                     |
| `{i in NODES: i <> "A"}`            | Set with condition                                  |
| `{(i,j) in ROUTES: cost[i,j] < 10}` | Filtered set of pairs                               |
| `setof {(i,j) in ROUTES} i`         | Set created from another indexed expression         |
| `{}`                                | Empty set                                           |


#### Naming rules

A set name — like any AMPL identifier — may consist of **letters**, **digits**, and the **underscore character** (`_`). It must not be a valid number (e.g., `123` is illegal as a name). Names are **case-sensitive**: `Prod`, `prod`, and `PROD` are three distinct identifiers.

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use ALL_CAPS for set names (e.g., <code>PROD</code>, <code>CITIES</code>, <code>TIME_PERIODS</code>). This visually distinguishes sets from parameters (typically lowercase) and variables (typically TitleCase), matching the convention used throughout the AMPL models.</div>

A small number of identifiers are **reserved words** in AMPL (e.g., `sum`, `in`, `within`) and cannot be used as set names. A larger group of names have predefined meanings (e.g., `prod` as the iterated-product operator) but can be redefined — though doing so is inadvisable.

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Using a reserved word as a set name (e.g., <code>set in;</code> or <code>set default;</code>)
causes a syntax error at model-load time. If you need a name that resembles a reserved word,
prefix or suffix it: <code>IN_NODES</code>, <code>DEFAULT_SET</code>.</div>

#### Reserved names

| | |  |
|---------|-------------|---------|
|`Current` | `Complements` | `Integer` |
| `solve_result_num` | `IN` | `contains` |
| `less` | `suffix` | `INOUT` |
| `default` | `logical` | `sum` |
| `Infinity` | `dimen` | `max` |
| `symbolic` | `Initial` | `div` |
| `min` | `table` | `LOCAL` |
| `else` | `option` | `then` |
| `OUT` | `environ` | `setof` |
| `union` | `all` | `exists` |
| `shell_exitcode` | `until` | `binary` |
| `forall` | `solve_exitcode` | `while` |
| `by` | `if` | `solve_message` |
| `within` | `check` | `in` |
| `solve_result` | | |

#### Working with sets via amplpy

The amplpy Python API provides direct access to AMPL sets. Use `ampl.set[name]` (equivalent to `ampl.get_set(name)`) to obtain an `amplpy.Set` object.  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1__1.ipynb)

In [ ]:
# Define AMPL set PROD and assign values from a Python list
ampl.eval("set PROD;")                     # Declare empty set PROD in AMPL
ampl.set["PROD"].set_values(["bands", "coils", "plate"])  # Populate with 3 products

members = list(ampl.set["PROD"].members()) # Retrieve set members as Python list
n = ampl.set["PROD"].size()                # Get cardinality (number of elements)

Key `amplpy.Set` methods:

| Method | Returns | Notes |
|---|---|---|
| `set_values(data)` | `None` | Accepts a list or DataFrame; replaces all current members |
| `members()` | `list` | All current members as Python objects |
| `get_values()` | `DataFrame` | Members as a pandas DataFrame |
| `size()` | `int` | Cardinality of the set |